## Read data and check contents

Download data, upload to dataframe and show five rows and name of the columns

In [ ]:
!wget -O wine.xlsx https://github.com/jordipereiragude/dataforcourses/raw/refs/heads/main/wine.xlsx

In [ ]:
import pandas

df=pandas.read_excel("wine.xlsx")
print("=======")
print(df.head())
print("=======")
print(df.dtypes)
print("=======")


## Visual exploration

Plot a scatter plot with AGST values as x-axis and Price as y-axis

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

#plt.scatter(df['AGST'], df['Price'], color="#A23333",edgecolor="black",linewidth=0.5)
plt.scatter(df['AGST'], df['Price'], color='blue', alpha=0.7)
plt.title('Scatter Plot: AGST vs Price')
plt.xlabel('AGST')
plt.ylabel('Price')
plt.grid(True)
#plt.grid(True,linestyle=':',color='#D3D3D3',linewidth=0.6,alpha=0.8)
plt.show()

If you need to create new plots, you can base them on previous plots (no need to use an LLM here).

For instance, we can change the previous plot to represent the relationship of `Price` with `Age`

## Basic Model

Build a regression model using statmodels and its formula interface. The formula for the model is:

Price ~ AGST

After fitting the model, report its summary


In [ ]:
import statsmodels.formula.api as smf

# Build and fit the OLS regression model
model = smf.ols(formula='Price ~ AGST', data=df)
results = model.fit()

# Display the model summary
print(results.summary())

Plot the regression line with its 95% confidence interval together with the observations for model `model`

use:

- color="#A23333",edgecolor="black",linewidth=0.5 to represent the dots of the observations
- linestyle=':',color='#D3D3D3',linewidth=0.6,alpha=0.8 for grid layout

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sort AGST values to get a smooth line plot
agst_range = np.linspace(df['AGST'].min(), df['AGST'].max(), 100)
predict_df = pandas.DataFrame({'AGST': agst_range})

# Obtain predictions and confidence intervals
predictions = results.get_prediction(predict_df)
pred_summary = predictions.summary_frame(alpha=0.05)

plt.figure(figsize=(8, 6))

# Scatter plot of observations
plt.scatter(df['AGST'], df['Price'], color='#A23333', edgecolor='black', linewidth=0.5, label='Observations', zorder=3)

# Regression line
plt.plot(agst_range, pred_summary['mean'], color='blue', label='Regression Line')

# 95% Confidence Interval band
plt.fill_between(agst_range, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], color='blue', alpha=0.15, label='95% CI')

plt.title('Regression line with 95% Confidence Interval')
plt.xlabel('AGST')
plt.ylabel('Price')
plt.grid(True, linestyle=':', color='#D3D3D3', linewidth=0.6, alpha=0.8)
plt.legend()
plt.show()

## Model with all variables

We start with a model like:

Price ~ AGST + Year + WinterRain + HarvestRain + Age

In [ ]:
import statsmodels.formula.api as smf

# Build and fit the OLS regression model
model = smf.ols(formula='Price ~  AGST + Year + WinterRain + HarvestRain + Age', data=df)
results = model.fit()

# Display the model summary
print(results.summary())

### Checking the model

create a heatmap plot with correlation between numerical variables in df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate correlation matrix for numerical columns
corr_matrix = df.select_dtypes(include=['float64', 'int64']).corr()

# Set up the matplotlib figure
plt.figure(figsize=(8, 6))

# Generate a heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)

# Set title
plt.title('Correlation Heatmap of Numerical Variables')
plt.show()

Modify the model to remove either Age or Year.

Price ~  AGST + WinterRain + HarvestRain + Age

In [ ]:
import statsmodels.formula.api as smf

# Build and fit the OLS regression model
model = smf.ols(formula='Price ~  AGST + WinterRain + HarvestRain + Age', data=df)
results = model.fit()

# Display the model summary
print(results.summary())

### Check for column redundancy and possible extreme observations


Report the VIF of the variables in model `model`

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Get the design matrix of independent variables (including the Intercept)
X = results.model.exog
names = results.model.exog_names

# Calculate VIF for each variable
vif_data = pd.DataFrame()
vif_data["Variable"] = names
vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]

print(vif_data)

Using model

Price ~ AGST

report the Cook's distance of each observation. Provide a plot of the Cook distance for each observation and a table of those observations that are considered to be too influential. The table should also report the estimated value of Price to make a comparison.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Fit the specified model
simple_model = smf.ols(formula='Price ~ AGST', data=df)
simple_results = simple_model.fit()

# Calculate influence statistics
influence = simple_results.get_influence()

# Get Cook's distance and p-values
cooks_d, p_vals = influence.cooks_distance

# Create a visualization of Cook's distances
n = len(df)
threshold = 4 / n

plt.figure(figsize=(10, 5))
plt.bar(range(n), cooks_d, color='#A23333', edgecolor='black', alpha=0.8)
plt.axhline(y=threshold, color='blue', linestyle='--', label=f'Threshold (4/n = {threshold:.3f})')
plt.title("Cook's Distance for Each Observation (Model: Price ~ AGST)")
plt.xlabel('Observation Index')
plt.ylabel("Cook's Distance")
plt.grid(True, linestyle=':', color='#D3D3D3', linewidth=0.6, alpha=0.8)
plt.legend()
plt.show()

# Identify influential observations
influential_idx = np.where(cooks_d > threshold)[0]

# Prepare comparison table
comparison_df = df.iloc[influential_idx].copy()
comparison_df['Estimated_Price'] = simple_results.predict(comparison_df)
comparison_df['Cooks_Distance'] = cooks_d[influential_idx]

print("--- Cook's Distance of all observations ---")
for idx, d in enumerate(cooks_d):
    print(f"Observation {idx}: {d:.4f}")

print("\n--- Highly Influential Observations (Cook's D > 4/n) ---")
print(comparison_df[['Year', 'Price', 'Estimated_Price', 'AGST', 'Cooks_Distance']].to_string(index=True))

### Analysis of the residuals

Consider:

Price ~ AGST + WinterRain + HarvestRain + Age

Make a Q-Q plot of the residuals

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Retrieve the residuals from the model: Price ~ AGST + WinterRain + HarvestRain + Age
# Note: "results" from cell HWoP_ZZ7rsO7 corresponds to this exact model formulation
residuals = results.resid

# Create the Q-Q plot
fig, ax = plt.subplots(figsize=(8, 6))
sm.qqplot(residuals, line='s', ax=ax)

# Customize plot aesthetics
ax.set_title("Normal Q-Q Plot of Residuals", fontsize=14)
ax.set_xlabel("Theoretical Quantiles", fontsize=12)
ax.set_ylabel("Sample Quantiles", fontsize=12)
ax.grid(True, linestyle=':', color='#D3D3D3', linewidth=0.6, alpha=0.8)

plt.show()

Create plots for each variable in the model (Price ~ AGST + WinterRain + HarvestRain + Age). The x-axis should represent the value of the variable, while the y-axis are the residuals of the model. Also, try to create a smoothed curve of the relationship between the variable and the residuals

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the independent variables from the model
predictors = ['AGST', 'WinterRain', 'HarvestRain', 'Age']
residuals = results.resid

# Create a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(predictors):
    # Plot residuals against the predictor with a smoothed LOWESS line
    # Using 'linewidths' to avoid Matplotlib parameter conflicts in sns.regplot
    sns.regplot(
        x=df[var],
        y=residuals,
        ax=axes[i],
        lowess=True,
        scatter_kws={'color': '#A23333', 'edgecolor': 'black', 'linewidths': 0.5, 'alpha': 0.8},
        line_kws={'color': 'blue', 'linewidth': 2, 'label': 'Smoothed Trend'}
    )
    axes[i].axhline(y=0, color='gray', linestyle='--', linewidth=1)
    axes[i].set_title(f'Residuals vs {var}', fontsize=12)
    axes[i].set_xlabel(var, fontsize=10)
    axes[i].set_ylabel('Residuals', fontsize=10)
    axes[i].grid(True, linestyle=':', color='#D3D3D3', linewidth=0.6, alpha=0.8)
    axes[i].legend()

plt.tight_layout()
plt.show()

## Alternatives (log transform)

Create a new variable logPrice that takes as its value the logarithm of variable Price

In [ ]:
import numpy as np

# Create logPrice containing the natural logarithm of Price
df['logPrice'] = np.log(df['Price'])

# Verify the new column was successfully added
print(df[['Price', 'logPrice']].head())

Model:

logPrice ~ AGST + WinterRain + HarvestRain + Age

In [ ]:
import statsmodels.formula.api as smf

modeloLog = smf.ols(formula='logPrice ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

print(modeloLog.summary())

## Wrap-up

Report mean absolute error of model

modelFull = smf.ols(formula='Price ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

and model

modelLog = smf.ols(formula='logPrice ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

considering variable `Price` (remember modelLog result should be exponentiated before calculating the mean absolute error)

Also report Mean Squared Error

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# Fit both models as specified
modelFull = smf.ols(formula='Price ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()
modelLog = smf.ols(formula='logPrice ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

# Get predictions
predFull = modelFull.predict(df)
# Exponentiate predictions from the log-linear model to bring them back to the original Price scale
predLog = np.exp(modelLog.predict(df))

# Calculate MAE
mae_full = np.mean(np.abs(df['Price'] - predFull))
mae_log = np.mean(np.abs(df['Price'] - predLog))

# Calculate MSE
mse_full = np.mean((df['Price'] - predFull) ** 2)
mse_log = np.mean((df['Price'] - predLog) ** 2)

# Display the results
print(f"--- Model Comparison on Original Price Scale ---\n")
print(f"[Model Full (Linear)]")
print(f"  Mean Absolute Error (MAE): {mae_full:.4f}")
print(f"  Mean Squared Error (MSE):  {mse_full:.4f}\n")
print(f"[Model Log (Log-Linear, Back-transformed)]")
print(f"  Mean Absolute Error (MAE): {mae_log:.4f}")
print(f"  Mean Squared Error (MSE):  {mse_log:.4f}")